In [1]:
# transferable code functions
import numpy as np

def padding(layer:np.ndarray, mode:str = 'zero'):


    padded_ly_size = (layer.shape[0]+2, layer.shape[1]+2)
    padded_ly = np.zeros(padded_ly_size)
    padded_ly[1:-1,1:-1] = layer
   
    if mode == 'continue':
        padded_ly[0,1:-1] = layer[0,:]
        padded_ly[-1,1:-1] = layer[-1,:]

        padded_ly[1:-1,0] = layer[:,0]
        padded_ly[1:-1,-1] = layer[:,-1]

        padded_ly[0,0] = layer[0,0]
        padded_ly[0,-1] = layer[0,-1]
        padded_ly[-1,0] = layer[-1,0]
        padded_ly[-1,-1] = layer[-1,-1]

    return(padded_ly)

In [ ]:
#todo I'd like to create a version of this that can have dynamically sized convolutional layers
import numpy as np
from dataclasses import dataclass

print("start")

@dataclass
class NN_Config:
    input_shape: tuple
    kernel_shape : tuple
    kernel_num: int
    conv_blocks: int
    conv_shape: tuple #this should be replaced by stride eventually?
    output_shape: tuple

    #todo make a more flexible model architechure similar to YOLO

    

class Layer:
    def __init__(self, activations: np.ndarray, z_values: np.ndarray, dims:int, shape: tuple):
        self.activations = activations
        self.z_values = z_values
        self.shape = shape
        self.dims = dims
    
    @classmethod
    def from_shape(cls, layer_size, dtype=np.float32, **kwargs):
        activations = np.zeros(layer_size, dtype=dtype)
        z_vals = np.zeros(layer_size, dtype=dtype)
        shape = activations.shape
        dims = len(shape)
        return cls(activations, z_vals, dims, shape)

class Weights:
    def __init__(self, weights: np.ndarray, w_type: str, ly_dim: int, shape: tuple):
        self.weights = weights
        self.shape = shape
        self.w_type = w_type
        self.ly_dim = ly_dim

    @classmethod
    def filters_from_shape(cls, kernel_shape, num_filters, type=np.float32, init=True, **kwargs):
        # replaced the the layer size with the number of filters.  
        # while each filter is applied to every prev_ly location according to the stride, 
        # the filters are unique to themselves, not the a particular layer location
        if init:
        #ly_weights = np.random.rand(prev_ly_size, curr_ly_size)
            ly_weights = np.random.uniform(-1,1, size=(*kernel_shape, num_filters))
        else:
            ly_weights = np.zeros(*kernel_shape, num_filters)

        w_type = "conv"
        ly_dim = len(kernel_shape)
        shape = ly_weights.shape
        return cls(ly_weights, w_type, ly_dim, shape)

    @classmethod
    def full_connected_from_shape(cls, prev_ly_shape, curr_ly_shape, dtype=np.float32, init=True, **kwargs):
        if init:
            #ly_weights = np.random.rand(prev_ly_size, curr_ly_size)
            ly_weights = np.random.uniform(-1,1, size=(prev_ly_shape, curr_ly_shape))
        else:
            ly_weights = np.zeros(shape=(prev_ly_shape, curr_ly_shape))
        w_type = "fc"
        ly_dim = len(curr_ly_shape)
        shape = ly_weights.shape
        
        return cls(ly_weights, w_type, ly_dim, shape)

    def from_dynamic_shape(cls, kernel_shape, prev_ly_shape, curr_ly_shape, dtype=np.float32, init=True, **kwargs):
        print("not yet implemented :(")
        print("might be impossible :( :(")


class Biases:
    def __init__(self, biases: np.ndarray, dims:int, shape:tuple):
        self.biases = biases
        self.dims = dims
        self.shape = shape

    @classmethod
    def from_shape(cls, curr_ly_shape:tuple, dtype=np.float32, init=True, **kwargs):
        if init:
            biases = np.random.rand(*curr_ly_shape)
        else:
            biases = np.zeros(curr_ly_shape)
        
        dims = len(curr_ly_shape)
        shape = curr_ly_shape

        return cls(biases, dims, shape)

    def from_dynamic_shape(cls, kernel_shape, prev_ly_shape, curr_ly_shape, dtype=np.float32, init=True, **kwargs):
        print("not yet implemented :(")

class NN:
    def __init__(self, config:NN_Config, layers: list, weights: list, biases: list):
        self.config = config
        self.layers = layers # layers hold both actual activations and z values
        self.weights = weights
        self.biases = biases

    @classmethod
    def create_network(cls, config:NN_Config, **kwargs):
        layer0 = ConvLayer.from_shape(config.input_shape)
        hidden_lys = []
        for i in range(config.conv_blocks):
            hidden_lys.append(ConvLayer.from_shape(config.conv_shape))
        layer_fn = SimpleLayer.create_layer(config.output_shape)
        layers = [layer0, *hidden_lys, layer_fn]

        biases =[]
        for i in range(1, len(layers), 1):
            biases.append(ConvBias.from_shape(np.shape(layers[i].activations), init=False))


        weights = []
        for i in range(1, len(layers), 1):
            weights.append(ConvWeights.from_shape(config.kernel_shape, np.shape(layers[i].activations), init=True))


        return cls(config, layers, weights, biases)


def forward(input_vals:np.ndarray, net:NN):
    net.layers[0] = input_vals


    for index in range(1, len(net.layers), 1):
        shape = net.layers[index].activations.shape
        new_z_ly = np.zeros(shape)
        new_ly   = np.zeros(shape)
        #net.z_layers[index] = np.dot(net.layers[(index-1)], net.weights[index-1]) + net.biases[index-1]
        #todo: find a preforment way to do this!
        for i in shape[0]:
            for j in shape[1]:
                new_z_ly[i,j] = net.layers[(index - 1)].activations[i,j]* net.weights[index-1].weights[i,j]
                



#todo:
    #figure out network autocreation
    #figure out padding algorrithm
    #figure out down sizing

start


In [3]:
import numpy as np
filters  = {"prev_ly_size":28,
            "kernel_size": 3,
            "stride" : 1}
l1 = np.zeros(shape=(filters["prev_ly_size"],))
l11 = np.zeros(shape=(filters["prev_ly_size"],))

l2 = []
l1[0] = 1
for index, values in enumerate(l1):
    if index%filters["stride"] == 0:
        l1[index] = 1


    if l1[index] == 1:
        for x in range(filters["kernel_size"]): 
            y=x+1
            try:
                l11[index + (y - filters["kernel_size"]//2)] = l11[index + (y - filters["kernel_size"]//2)] + 1
            except IndexError as error:
                print("had an index error, continuing")
print(l1,"\n",l11)

had an index error, continuing
had an index error, continuing
had an index error, continuing
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1.] 
 [1. 2. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3.
 3. 3. 3. 3.]


In [46]:
size = 3
for x in range(size):

    print((x - size//2))

-1
0
1


In [18]:
9%3

0

In [ ]:
config = NN_Config(
    kernel_shape = (3,3),
    input_shape=(28,28),
    kernel_num=12,
    conv_blocks=3,
    conv_shape=(28,28),
    output_shape = (28,28)
    )

net = NN.create_network(config)
